# BMRCL Demand Prediction — Modeling Dataset & Temporal Split

**Task:** predict the number of passengers boarding a given Namma Metro station during the **next hour**.

\[
y_{s,t}=D_{s,t+1}
\]

This notebook converts the validated raw data into reproducible modeling datasets. It does **not** train models yet.

### Main decisions
- Primary target: next-hour station boarding demand.
- Study period: **2025-08-11 through 2025-09-30**.
- The Aug 19–31 source-data gap is preserved; it is never imputed as zero.
- Historical lags are created by exact timestamp lookup.
- Current-hour weather is used; observed future weather is not.
- Chronological split:
  - Train: Aug 11–Sep 14
  - Validation: Sep 15–Sep 21
  - Test: Sep 22–Sep 30


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

candidates = [Path("Data"), Path("../Data"), Path("/mnt/data/BMRCL_Data/Data")]
DATA_DIR = next((p for p in candidates if p.exists() and p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Data directory not found. Put the notebook in the repository root or edit DATA_DIR.")

PROJECT_ROOT = DATA_DIR.parent
OUT = PROJECT_ROOT / "processed_modeling"
OUT.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_DIR.resolve())
print("Output:", OUT.resolve())


## 1. Load the canonical raw files

In [ ]:
def read_csv(name, sep=","):
    return pd.read_csv(DATA_DIR / name, sep=sep)

station_hourly = read_csv("station-hourly.csv", ";")
weather = read_csv("Bengaluru_AUG-SEPT_Weather.csv")
calendar = read_csv("bengaluru_calendar_aug_sep_2025.csv")
parking = read_csv("metro_station_parking_details.csv")
station_names = read_csv("station-names.csv", ";")

print("station-hourly:", station_hourly.shape)
print("weather:", weather.shape)
print("calendar:", calendar.shape)
print("parking:", parking.shape)


## 2. Normalize stations and create the base table

In [ ]:
alias_map = dict(zip(
    station_names["alt_name"].astype(str).str.strip(),
    station_names["name"].astype(str).str.strip()
))

station_hourly["Station"] = (
    station_hourly["Station"].astype(str).str.strip().replace(alias_map)
)
station_hourly["Date"] = pd.to_datetime(station_hourly["Date"], errors="coerce")
station_hourly["Hour"] = pd.to_numeric(station_hourly["Hour"], errors="coerce").astype(int)
station_hourly["Ridership"] = pd.to_numeric(station_hourly["Ridership"], errors="coerce")

station_hourly["datetime"] = (
    station_hourly["Date"] + pd.to_timedelta(station_hourly["Hour"], unit="h")
)

START = pd.Timestamp("2025-08-11")
END = pd.Timestamp("2025-09-30 23:00:00")

demand = station_hourly.loc[
    station_hourly["datetime"].between(START, END),
    ["datetime", "Date", "Hour", "Station", "Ridership"]
].rename(columns={"Station": "station", "Ridership": "demand"})

demand = demand.sort_values(["station", "datetime"]).reset_index(drop=True)

print("Rows:", f"{len(demand):,}")
print("Stations:", demand["station"].nunique())
print("Duplicates:", demand.duplicated(["station", "datetime"]).sum())
print("Negative demand:", (demand["demand"] < 0).sum())


## 3. Add calendar features

In [ ]:
calendar["date"] = pd.to_datetime(calendar["date"], errors="coerce")

calendar_cols = [
    "date", "day_of_week", "is_weekend", "is_public_holiday",
    "is_major_city_event", "is_working_day_proxy",
    "is_day_before_public_holiday", "is_day_after_public_holiday"
]

base = demand.merge(
    calendar[calendar_cols].rename(columns={"date": "Date"}),
    on="Date",
    how="left",
    validate="many_to_one"
)

base["day_of_week_num"] = base["datetime"].dt.dayofweek
base["month_num"] = base["datetime"].dt.month
base["hour_sin"] = np.sin(2 * np.pi * base["Hour"] / 24)
base["hour_cos"] = np.cos(2 * np.pi * base["Hour"] / 24)

print("Calendar missing rows:", base["is_weekend"].isna().sum())


## 4. Add current-hour weather features

In [ ]:
weather["datetime"] = pd.to_datetime(
    dict(
        year=weather["year"],
        month=weather["month"],
        day=weather["day"],
        hour=weather["hour"]
    ),
    errors="coerce"
)

weather_cols = ["datetime", "temp", "rhum", "prcp", "wspd", "pres", "cldc", "coco"]

base = base.merge(
    weather[weather_cols],
    on="datetime",
    how="left",
    validate="many_to_one"
)

print("Weather coverage:", f"{100 * base['temp'].notna().mean():.2f}%")


## 5. Add station-static parking features

In [ ]:
parking["station_normalized"] = (
    parking["NAME_OF_STATIONS"].astype(str).str.strip().replace(alias_map)
)

# Convert likely capacity columns to numeric.
for col in parking.columns:
    if col not in ["NAME_OF_STATIONS", "station_normalized"]:
        parking[col] = pd.to_numeric(
            parking[col].astype(str).str.replace(",", "", regex=False),
            errors="coerce"
        )

parking_cols = [
    c for c in parking.columns
    if c not in ["NAME_OF_STATIONS", "station_normalized"]
    and any(k in c.lower() for k in ["2-wheeler", "4-wheeler", "cycle", "lcv", "total"])
]

print("Parking columns:", parking_cols)

if parking_cols:
    parking_model = (
        parking.groupby("station_normalized")[parking_cols]
        .max()
        .reset_index()
        .rename(columns={"station_normalized": "station"})
    )
    base = base.merge(
        parking_model,
        on="station",
        how="left",
        validate="many_to_one"
    )


## 6. Create exact historical lags

The gap from Aug 19–31 means normal row-based `shift(24)` or `shift(168)` is unsafe.

Every lag below is therefore created by exact timestamp lookup:

\[
Lag_k(s,t)=D_{s,t-k}
\]

If the exact timestamp is absent, the lag remains missing.


In [ ]:
lookup = demand[["station", "datetime", "demand"]].copy()

def add_exact_lag(df, lag_hours):
    x = lookup.copy()
    x["datetime"] = x["datetime"] + pd.Timedelta(hours=lag_hours)
    x = x.rename(columns={"demand": f"demand_lag_{lag_hours}h"})
    return df.merge(
        x[["station", "datetime", f"demand_lag_{lag_hours}h"]],
        on=["station", "datetime"],
        how="left",
        validate="one_to_one"
    )

for lag in [1, 2, 3, 24, 168]:
    base = add_exact_lag(base, lag)

print(
    base[
        ["demand_lag_1h", "demand_lag_2h", "demand_lag_3h",
         "demand_lag_24h", "demand_lag_168h"]
    ].isna().mean().mul(100).round(2)
)


## 7. Create leakage-safe rolling demand features

In [ ]:
roll = (
    demand[["station", "datetime", "demand"]]
    .sort_values(["station", "datetime"])
)

for window, name in [("3h", "rolling_mean_3h"),
                     ("6h", "rolling_mean_6h"),
                     ("24h", "rolling_mean_24h")]:
    r = (
        roll.set_index("datetime")
        .groupby("station")["demand"]
        .rolling(window, min_periods=1)
        .mean()
        .rename(name)
        .reset_index()
    )
    base = base.merge(
        r,
        on=["station", "datetime"],
        how="left",
        validate="one_to_one"
    )


## 8. Create the next-hour target

In [ ]:
target = demand[["station", "datetime", "demand"]].copy()
target["datetime"] = target["datetime"] - pd.Timedelta(hours=1)
target = target.rename(columns={"demand": "target_next_hour"})

base = base.merge(
    target[["station", "datetime", "target_next_hour"]],
    on=["station", "datetime"],
    how="left",
    validate="one_to_one"
)

print("Target coverage:", f"{100 * base['target_next_hour'].notna().mean():.2f}%")
print("Missing targets:", f"{base['target_next_hour'].isna().sum():,}")


## 9. Explicit checks for the August data gap

In [ ]:
sep1 = base[base["Date"] == pd.Timestamp("2025-09-01")]

print("Sep 1 rows:", len(sep1))
print("Sep 1 lag-24 non-null:", sep1["demand_lag_24h"].notna().sum())
print("Sep 1 lag-168 non-null:", sep1["demand_lag_168h"].notna().sum())

assert sep1["demand_lag_24h"].isna().all()
assert sep1["demand_lag_168h"].isna().all()

print("PASS: the Aug 19–31 gap is not being incorrectly bridged.")


## 10. Define experimental feature groups

In [ ]:
TIME = [
    "station", "Hour", "day_of_week_num", "month_num",
    "hour_sin", "hour_cos"
]

CALENDAR = [
    "is_weekend", "is_public_holiday", "is_major_city_event",
    "is_working_day_proxy", "is_day_before_public_holiday",
    "is_day_after_public_holiday"
]

WEATHER = ["temp", "rhum", "prcp", "wspd", "pres", "cldc", "coco"]

HISTORY = [
    "demand_lag_1h", "demand_lag_2h", "demand_lag_3h",
    "demand_lag_24h", "demand_lag_168h",
    "rolling_mean_3h", "rolling_mean_6h", "rolling_mean_24h"
]

STATIC = [
    c for c in base.columns
    if any(k in c.lower() for k in ["2-wheeler", "4-wheeler", "cycle", "lcv", "total"])
]

TARGET = "target_next_hour"

feature_sets = {
    "E1_time_station": TIME,
    "E2_time_calendar": TIME + CALENDAR,
    "E3_time_calendar_weather": TIME + CALENDAR + WEATHER,
    "E4_full_temporal": TIME + CALENDAR + WEATHER + HISTORY,
    "E5_full_plus_static": TIME + CALENDAR + WEATHER + HISTORY + STATIC,
}

for name, features in feature_sets.items():
    print(f"{name}: {len(features)} features")


## 11. Chronological train / validation / test split

In [ ]:
TRAIN_END = pd.Timestamp("2025-09-14 23:00:00")
VAL_START = pd.Timestamp("2025-09-15")
VAL_END = pd.Timestamp("2025-09-21 23:00:00")
TEST_START = pd.Timestamp("2025-09-22")
TEST_END = pd.Timestamp("2025-09-30 23:00:00")

base["split"] = np.select(
    [
        base["datetime"] <= TRAIN_END,
        base["datetime"].between(VAL_START, VAL_END),
        base["datetime"].between(TEST_START, TEST_END),
    ],
    ["train", "validation", "test"],
    default="outside"
)

display(
    base.groupby("split")["datetime"]
    .agg(["min", "max", "count"])
)


## 12. Count usable samples for each experiment

In [ ]:
def readiness_table(df, features):
    rows = []
    for split in ["train", "validation", "test"]:
        part = df[df["split"] == split]
        required = list(features) + [TARGET]
        usable = part[required].notna().all(axis=1)

        rows.append({
            "split": split,
            "raw_rows": len(part),
            "usable_rows": int(usable.sum()),
            "dropped_rows": int((~usable).sum()),
            "usable_pct": round(100 * usable.mean(), 2)
        })

    return pd.DataFrame(rows)

readiness = []

for name, features in feature_sets.items():
    x = readiness_table(base, features)
    x.insert(0, "experiment", name)
    readiness.append(x)

readiness = pd.concat(readiness, ignore_index=True)
display(readiness)


## 13. Save experiment-ready datasets

In [ ]:
exp_dir = OUT / "experiments"
exp_dir.mkdir(parents=True, exist_ok=True)

for name, features in feature_sets.items():
    cols = ["datetime", "Date", "station", TARGET] + [
        c for c in features if c != "station"
    ]
    cols = [c for c in cols if c in base.columns]

    complete = base.dropna(subset=list(features) + [TARGET]).copy()

    for split in ["train", "validation", "test"]:
        path = exp_dir / f"{name}_{split}.csv"
        complete.loc[complete["split"] == split, cols].to_csv(path, index=False)
        print(path.name, ":", len(complete.loc[complete["split"] == split]), "rows")

base.to_csv(OUT / "bmrcl_modeling_master.csv", index=False)
readiness.to_csv(OUT / "experiment_readiness.csv", index=False)

print("\nSaved master and experiment datasets to:", OUT.resolve())


## 14. Final leakage checks

In [ ]:
# Target must be exact t+1 demand.
lookup_dict = dict(zip(
    zip(demand["station"], demand["datetime"]),
    demand["demand"]
))

sample = base[base[TARGET].notna()].sample(
    min(1000, base[TARGET].notna().sum()),
    random_state=42
)

recomputed = np.array([
    lookup_dict.get((s, t + pd.Timedelta(hours=1)), np.nan)
    for s, t in zip(sample["station"], sample["datetime"])
])

assert np.array_equal(sample[TARGET].values, recomputed)

# Future weather fields are not used.
all_features = [f for group in feature_sets.values() for f in group]
assert not any("future" in f.lower() or "next" in f.lower() for f in all_features)

# Derived OD revenue is not used.
assert "Revenue" not in all_features

# Train < validation < test in time.
assert base.loc[base["split"] == "train", "datetime"].max() <        base.loc[base["split"] == "validation", "datetime"].min()
assert base.loc[base["split"] == "validation", "datetime"].max() <        base.loc[base["split"] == "test", "datetime"].min()

print("PASS: target alignment, future-feature, revenue, and chronological-split checks all passed.")


## What comes next

The saved datasets are now fixed inputs for the modeling notebook.

Recommended model progression:

1. Last-observation baseline.
2. Historical/same-hour baseline.
3. Linear/Ridge regression.
4. Random Forest.
5. XGBoost.
6. LSTM/GRU only after the tabular baselines are established.
7. Optional OD/spatial experiment afterward.

The main paper question can then become:

> **How much does each information group—time, calendar, weather, historical demand, and station characteristics—improve one-hour-ahead BMRCL station boarding-demand prediction?**
